# Black-Scholes: SymPy Derivations

All results from `02-pde-solution.md` are **derived or verified symbolically** here:
+ the key identity
+ BS PDE
+ Greeks (including Voma)
+ the put-call parity


In [1]:
import sympy as sp

sp.init_printing(use_unicode=True)

# --- symbols (all positive so log/sqrt are real) ---
S, K, T, r, q, sigma = sp.symbols('S K T r q sigma', positive=True)
x = sp.Symbol('x', real=True)

# --- d1 and d2 ---
d1 = (sp.log(S/K) + (r - q + sigma**2/2)*T) / (sigma*sp.sqrt(T))
d2 = d1 - sigma*sp.sqrt(T)

# --- standard normal CDF and PDF ---
def N(z):
    """Standard normal CDF."""
    return sp.Rational(1, 2) * (1 + sp.erf(z / sp.sqrt(2)))

def n(z):
    """Standard normal PDF."""
    return sp.exp(-z**2 / 2) / sp.sqrt(2 * sp.pi)

# --- call and put prices ---
C = S*sp.exp(-q*T)*N(d1) - K*sp.exp(-r*T)*N(d2)
P = K*sp.exp(-r*T)*N(-d2) - S*sp.exp(-q*T)*N(-d1)

print('d1 =')
display(d1)
print('d2 =')
display(d2)
print('C =')
display(C)

d1 =


  ⎛          2⎞         
  ⎜         σ ⎟      ⎛S⎞
T⋅⎜-q + r + ──⎟ + log⎜─⎟
  ⎝         2 ⎠      ⎝K⎠
────────────────────────
          √T⋅σ          

d2 =


          ⎛          2⎞         
          ⎜         σ ⎟      ⎛S⎞
        T⋅⎜-q + r + ──⎟ + log⎜─⎟
          ⎝         2 ⎠      ⎝K⎠
-√T⋅σ + ────────────────────────
                  √T⋅σ          

C =


    ⎛   ⎛   ⎛          ⎛          2⎞         ⎞⎞    ⎞                           ↪
    ⎜   ⎜   ⎜          ⎜         σ ⎟      ⎛S⎞⎟⎟    ⎟                           ↪
    ⎜   ⎜   ⎜        T⋅⎜-q + r + ──⎟ + log⎜─⎟⎟⎟    ⎟           ⎛   ⎛   ⎛  ⎛    ↪
    ⎜   ⎜   ⎜          ⎝         2 ⎠      ⎝K⎠⎟⎟    ⎟           ⎜   ⎜   ⎜  ⎜    ↪
    ⎜   ⎜√2⋅⎜-√T⋅σ + ────────────────────────⎟⎟    ⎟           ⎜   ⎜√2⋅⎜T⋅⎜-q  ↪
    ⎜   ⎜   ⎝                  √T⋅σ          ⎠⎟    ⎟           ⎜   ⎜   ⎝  ⎝    ↪
    ⎜erf⎜─────────────────────────────────────⎟    ⎟           ⎜erf⎜────────── ↪
    ⎜   ⎝                  2                  ⎠   1⎟  -T⋅r     ⎜   ⎝           ↪
- K⋅⎜────────────────────────────────────────── + ─⎟⋅ℯ     + S⋅⎜────────────── ↪
    ⎝                    2                        2⎠           ⎝               ↪

↪                                
↪                                
↪        2⎞         ⎞⎞    ⎞      
↪       σ ⎟      ⎛S⎞⎟⎟    ⎟      
↪ + r + ──⎟ + log⎜─⎟⎟⎟    ⎟      
↪       2 ⎠      ⎝K

## 1  Key identity

In [2]:
# Claim:  S·e^{-qT}·n(d1) = K·e^{-rT}·n(d2)
#
# Strategy: expand d1² - d2² = (d1-d2)(d1+d2).
# Since d1 - d2 = σ√T, we get d1² - d2² = 2·ln(S/K) + 2(r-q)T.
# Therefore n(d1)/n(d2) = exp(-(d1²-d2²)/2) = (K/S)·e^{-(r-q)T},
# which rearranges to the identity. SymPy verifies directly:

diff_sq = sp.expand(d1**2 - d2**2)
print('d1² - d2² =')
display(sp.simplify(diff_sq))

key_diff = sp.simplify(S*sp.exp(-q*T)*n(d1) - K*sp.exp(-r*T)*n(d2))
print(f'\nS·e^(-qT)·n(d1) - K·e^(-rT)·n(d2)  =  {key_diff}')
assert key_diff == 0, 'key identity failed'

d1² - d2² =


-2⋅T⋅q + 2⋅T⋅r - 2⋅log(K) + 2⋅log(S)


S·e^(-qT)·n(d1) - K·e^(-rT)·n(d2)  =  0


## 2  BS PDE

In [3]:
# With T = time-to-maturity τ, the BS PDE reads:
#   ∂C/∂τ = ½σ²S²·∂²C/∂S² + (r-q)S·∂C/∂S - rC
# i.e. the residual  ∂C/∂τ - ½σ²S²Γ - (r-q)SΔ + rC  should be zero.

pde_lhs = sp.diff(C, T)
pde_rhs = (sp.Rational(1,2)*sigma**2*S**2*sp.diff(C, S, 2)
           + (r - q)*S*sp.diff(C, S)
           - r*C)

residual = sp.simplify(pde_lhs - pde_rhs)
print(f'PDE residual  =  {residual}')
assert residual == 0, 'PDE not satisfied'

PDE residual  =  0


## 3  Partial derivatives of d₁ and d₂

In [4]:
# These building blocks appear in every Greek derivation.

dd1_dS    = sp.diff(d1, S)
dd2_dS    = sp.diff(d2, S)
dd1_dsig  = sp.diff(d1, sigma)
dd2_dsig  = sp.diff(d2, sigma)
dd1_dT    = sp.diff(d1, T)
dd2_dT    = sp.diff(d2, T)

print('∂d1/∂S =')
display(sp.simplify(dd1_dS))

# d1 and d2 share the same S-derivative
print('∂d2/∂S - ∂d1/∂S =', sp.simplify(dd2_dS - dd1_dS))

print('\n∂d1/∂σ =')
display(sp.simplify(dd1_dsig))
print('∂d1/∂σ + d2/σ =', sp.simplify(dd1_dsig + d2/sigma))   # should be 0

print('\n∂d2/∂σ =')
display(sp.simplify(dd2_dsig))
print('∂d2/∂σ + d1/σ =', sp.simplify(dd2_dsig + d1/sigma))   # should be 0

print('\n∂d1/∂T =')
display(sp.simplify(dd1_dT))

∂d1/∂S =


  1   
──────
S⋅√T⋅σ

∂d2/∂S - ∂d1/∂S = 0

∂d1/∂σ =


                   2                      
2⋅T⋅q - 2⋅T⋅r + T⋅σ  + 2⋅log(K) - 2⋅log(S)
──────────────────────────────────────────
                       2                  
                 2⋅√T⋅σ                   

∂d1/∂σ + d2/σ = 0

∂d2/∂σ =


    ⎛              2⎞                  
  T⋅⎝-2⋅q + 2⋅r + σ ⎠                  
- ─────────────────── + log(K) - log(S)
           2                           
───────────────────────────────────────
                     2                 
                 √T⋅σ                  

∂d2/∂σ + d1/σ = 0

∂d1/∂T =


                         ⎛ 2⎞
  ⎛              2⎞      ⎜S ⎟
T⋅⎝-2⋅q + 2⋅r + σ ⎠ - log⎜──⎟
                         ⎜ 2⎟
                         ⎝K ⎠
─────────────────────────────
             3/2             
          4⋅T   ⋅σ           

## 4  Greeks

Each cell:
1. computes the raw symbolic derivative with `sp.diff`,
2. proposes a compact closed form (as a SymPy expression),
3. verifies equality via `sp.simplify`.

In [5]:
# --- Delta = ∂C/∂S ---
# Raw derivative: N(d1)·e^{-qT} term, plus two n(d_i)·∂d_i/∂S terms
# that cancel by the key identity.

Delta_raw     = sp.diff(C, S)
Delta_formula = sp.exp(-q*T) * N(d1)

check = sp.simplify(Delta_raw - Delta_formula)
print(f'Δ  =  {sp.simplify(Delta_formula)}')
print(f'simplify(raw - formula) = {check}')
assert check == 0

Δ  =  (erf(sqrt(2)*(T*(-2*q + 2*r + sigma**2) + log(S**2/K**2))/(4*sqrt(T)*sigma)) + 1)*exp(-T*q)/2
simplify(raw - formula) = 0


In [6]:
# --- Gamma = ∂²C/∂S² = ∂Delta/∂S ---
# Differentiating e^{-qT}·N(d1) once more in S:
#   ∂/∂S [e^{-qT}·N(d1)] = e^{-qT}·n(d1)·∂d1/∂S
# with ∂d1/∂S = 1/(S·σ√T).

Gamma_raw     = sp.diff(C, S, 2)
Gamma_formula = sp.exp(-q*T) * n(d1) / (S * sigma * sp.sqrt(T))

check = sp.simplify(Gamma_raw - Gamma_formula)
print('Γ  =')
display(Gamma_formula)
print(f'simplify(raw - formula) = {check}')
assert check == 0

Γ  =


                                     2 
           ⎛  ⎛          2⎞         ⎞  
           ⎜  ⎜         σ ⎟      ⎛S⎞⎟  
          -⎜T⋅⎜-q + r + ──⎟ + log⎜─⎟⎟  
           ⎝  ⎝         2 ⎠      ⎝K⎠⎠  
          ─────────────────────────────
                          2            
    -T⋅q             2⋅T⋅σ             
√2⋅ℯ    ⋅ℯ                             
───────────────────────────────────────
              2⋅√π⋅S⋅√T⋅σ              

simplify(raw - formula) = 0


In [7]:
# --- Vega = ∂C/∂σ ---
# Both N(d1) and N(d2) are differentiated via n(d_i)·∂d_i/∂σ.
# Using ∂d1/∂σ = -d2/σ and ∂d2/∂σ = -d1/σ, and the key identity,
# the expression collapses to S·e^{-qT}·n(d1)·√T.

Vega_raw     = sp.diff(C, sigma)
Vega_formula = S * sp.exp(-q*T) * n(d1) * sp.sqrt(T)

check = sp.simplify(Vega_raw - Vega_formula)
print('ν  =')
display(Vega_formula)
print(f'simplify(raw - formula) = {check}')
assert check == 0

ν  =


                                          2 
                ⎛  ⎛          2⎞         ⎞  
                ⎜  ⎜         σ ⎟      ⎛S⎞⎟  
               -⎜T⋅⎜-q + r + ──⎟ + log⎜─⎟⎟  
                ⎝  ⎝         2 ⎠      ⎝K⎠⎠  
               ─────────────────────────────
                               2            
         -T⋅q             2⋅T⋅σ             
√2⋅S⋅√T⋅ℯ    ⋅ℯ                             
────────────────────────────────────────────
                    2⋅√π                    

simplify(raw - formula) = 0


In [8]:
# --- Voma = ∂ν/∂σ = ∂²C/∂σ² ---
#
# Step 1: derive n'(x) = -x·n(x) from SymPy.
n_prime_check = sp.simplify(sp.diff(n(x), x) + x*n(x))
print(f"n'(x) + x·n(x) = {n_prime_check}  (must be 0)")
assert n_prime_check == 0

# Step 2: ∂d1/∂σ = -d2/σ  (verified in section 3).

# Step 3: chain rule on ν = S·e^{-qT}·n(d1)·√T:
#   ∂ν/∂σ = S·e^{-qT}·√T · n'(d1) · ∂d1/∂σ
#          = S·e^{-qT}·√T · (-d1·n(d1)) · (-d2/σ)
#          = ν · d1·d2/σ

nu = S * sp.exp(-q*T) * n(d1) * sp.sqrt(T)   # = Vega_formula
Voma_formula = nu * d1 * d2 / sigma

Voma_raw = sp.diff(C, sigma, 2)

check = sp.simplify(Voma_raw - Voma_formula)
print('\nVoma  =')
display(sp.simplify(Voma_formula))
print(f'simplify(raw - formula) = {check}')
assert check == 0

n'(x) + x·n(x) = 0  (must be 0)

Voma  =


                                                                               ↪
                                                                               ↪
                                                                               ↪
                                                                               ↪
                                                                               ↪
                                                                               ↪
                                                                               ↪
     ⎛                         ⎛ 2⎞⎞ ⎛                                    ⎛ 2⎞ ↪
     ⎜  ⎛              2⎞      ⎜S ⎟⎟ ⎜       2     ⎛              2⎞      ⎜S ⎟ ↪
√2⋅S⋅⎜T⋅⎝-2⋅q + 2⋅r + σ ⎠ + log⎜──⎟⎟⋅⎜- 2⋅T⋅σ  + T⋅⎝-2⋅q + 2⋅r + σ ⎠ + log⎜──⎟ ↪
     ⎜                         ⎜ 2⎟⎟ ⎜                                    ⎜ 2⎟ ↪
     ⎝                         ⎝K ⎠⎠ ⎝                                    ⎝K ⎠ ↪
────────────────────────────

simplify(raw - formula) = 0


In [9]:
# --- Theta = -∂C/∂t = +∂C/∂T  (market convention: negative of calendar-time deriv) ---
#
# ∂C/∂T differentiates both the e^{±·T} discount factors and the N(d_i) via n(d_i)·∂d_i/∂T.
# After the key identity absorbs the PDF cross-terms the result is:
#
#   ∂C/∂T = S·e^{-qT}·n(d1)·σ/(2√T) + r·K·e^{-rT}·N(d2) - q·S·e^{-qT}·N(d1)
#
# Market Theta  Θ = -∂C/∂t = -(the expression above), i.e. it is negative for calls.

dC_dT_raw = sp.diff(C, T)   # positive quantity

dC_dT_formula = (S*sp.exp(-q*T)*n(d1)*sigma / (2*sp.sqrt(T))
                 + r*K*sp.exp(-r*T)*N(d2)
                 - q*S*sp.exp(-q*T)*N(d1))

Theta_formula = -dC_dT_formula   # market convention

check = sp.simplify(dC_dT_raw - dC_dT_formula)
print('∂C/∂T  =')
display(dC_dT_formula)
print('\nΘ = -∂C/∂t = -∂C/∂T  =  (negative for calls)')
display(Theta_formula)
print(f'simplify(raw - formula) = {check}')
assert check == 0

∂C/∂T  =


                                                                               ↪
    ⎛   ⎛   ⎛          ⎛          2⎞         ⎞⎞    ⎞                           ↪
    ⎜   ⎜   ⎜          ⎜         σ ⎟      ⎛S⎞⎟⎟    ⎟                           ↪
    ⎜   ⎜   ⎜        T⋅⎜-q + r + ──⎟ + log⎜─⎟⎟⎟    ⎟             ⎛   ⎛   ⎛  ⎛  ↪
    ⎜   ⎜   ⎜          ⎝         2 ⎠      ⎝K⎠⎟⎟    ⎟             ⎜   ⎜   ⎜  ⎜  ↪
    ⎜   ⎜√2⋅⎜-√T⋅σ + ────────────────────────⎟⎟    ⎟             ⎜   ⎜√2⋅⎜T⋅⎜- ↪
    ⎜   ⎜   ⎝                  √T⋅σ          ⎠⎟    ⎟             ⎜   ⎜   ⎝  ⎝  ↪
    ⎜erf⎜─────────────────────────────────────⎟    ⎟             ⎜erf⎜──────── ↪
    ⎜   ⎝                  2                  ⎠   1⎟  -T⋅r       ⎜   ⎝         ↪
K⋅r⋅⎜────────────────────────────────────────── + ─⎟⋅ℯ     - S⋅q⋅⎜──────────── ↪
    ⎝                    2                        2⎠             ⎝             ↪

↪                                                                              ↪
↪                          


Θ = -∂C/∂t = -∂C/∂T  =  (negative for calls)


                                                                               ↪
      ⎛   ⎛   ⎛          ⎛          2⎞         ⎞⎞    ⎞                         ↪
      ⎜   ⎜   ⎜          ⎜         σ ⎟      ⎛S⎞⎟⎟    ⎟                         ↪
      ⎜   ⎜   ⎜        T⋅⎜-q + r + ──⎟ + log⎜─⎟⎟⎟    ⎟             ⎛   ⎛   ⎛   ↪
      ⎜   ⎜   ⎜          ⎝         2 ⎠      ⎝K⎠⎟⎟    ⎟             ⎜   ⎜   ⎜   ↪
      ⎜   ⎜√2⋅⎜-√T⋅σ + ────────────────────────⎟⎟    ⎟             ⎜   ⎜√2⋅⎜T⋅ ↪
      ⎜   ⎜   ⎝                  √T⋅σ          ⎠⎟    ⎟             ⎜   ⎜   ⎝   ↪
      ⎜erf⎜─────────────────────────────────────⎟    ⎟             ⎜erf⎜────── ↪
      ⎜   ⎝                  2                  ⎠   1⎟  -T⋅r       ⎜   ⎝       ↪
- K⋅r⋅⎜────────────────────────────────────────── + ─⎟⋅ℯ     + S⋅q⋅⎜────────── ↪
      ⎝                    2                        2⎠             ⎝           ↪

↪                                                                              ↪
↪                          

simplify(raw - formula) = 0


In [10]:
# --- Rho = ∂C/∂r ---
# r appears in both the N(d_i) arguments and the K·e^{-rT} discount.
# The n(d_i)·∂d_i/∂r terms cancel again by the key identity, leaving
# only the explicit discount derivative: K·T·e^{-rT}·N(d2).

Rho_raw     = sp.diff(C, r)
Rho_formula = K * T * sp.exp(-r*T) * N(d2)

check = sp.simplify(Rho_raw - Rho_formula)
print('ρ  =')
display(Rho_formula)
print(f'simplify(raw - formula) = {check}')
assert check == 0

ρ  =


    ⎛   ⎛   ⎛          ⎛          2⎞         ⎞⎞    ⎞      
    ⎜   ⎜   ⎜          ⎜         σ ⎟      ⎛S⎞⎟⎟    ⎟      
    ⎜   ⎜   ⎜        T⋅⎜-q + r + ──⎟ + log⎜─⎟⎟⎟    ⎟      
    ⎜   ⎜   ⎜          ⎝         2 ⎠      ⎝K⎠⎟⎟    ⎟      
    ⎜   ⎜√2⋅⎜-√T⋅σ + ────────────────────────⎟⎟    ⎟      
    ⎜   ⎜   ⎝                  √T⋅σ          ⎠⎟    ⎟      
    ⎜erf⎜─────────────────────────────────────⎟    ⎟      
    ⎜   ⎝                  2                  ⎠   1⎟  -T⋅r
K⋅T⋅⎜────────────────────────────────────────── + ─⎟⋅ℯ    
    ⎝                    2                        2⎠      

simplify(raw - formula) = 0


In [11]:
# --- Summary table (compact closed forms) ---

greeks = [
    ('Δ  (Delta)',  Delta_formula),
    ('Γ  (Gamma)',  Gamma_formula),
    ('ν  (Vega)',   Vega_formula),
    ('Voma',        Voma_formula),
    ('Θ  (Theta)',  Theta_formula),
    ('ρ  (Rho)',    Rho_formula),
]

for name, formula in greeks:
    print(f'{name} =')
    display(sp.simplify(formula))
    print()

Δ  (Delta) =


⎛   ⎛   ⎛                         ⎛ 2⎞⎞⎞    ⎞      
⎜   ⎜   ⎜  ⎛              2⎞      ⎜S ⎟⎟⎟    ⎟      
⎜   ⎜√2⋅⎜T⋅⎝-2⋅q + 2⋅r + σ ⎠ + log⎜──⎟⎟⎟    ⎟      
⎜   ⎜   ⎜                         ⎜ 2⎟⎟⎟    ⎟      
⎜   ⎜   ⎝                         ⎝K ⎠⎠⎟    ⎟  -T⋅q
⎜erf⎜──────────────────────────────────⎟ + 1⎟⋅ℯ    
⎝   ⎝              4⋅√T⋅σ              ⎠    ⎠      
───────────────────────────────────────────────────
                         2                         


Γ  (Gamma) =


                                          2
           ⎛                         ⎛ 2⎞⎞ 
           ⎜  ⎛              2⎞      ⎜S ⎟⎟ 
           ⎜T⋅⎝-2⋅q + 2⋅r + σ ⎠ + log⎜──⎟⎟ 
           ⎜                         ⎜ 2⎟⎟ 
           ⎝                         ⎝K ⎠⎠ 
    -T⋅q - ────────────────────────────────
                             2             
                        8⋅T⋅σ              
√2⋅ℯ                                       
───────────────────────────────────────────
                2⋅√π⋅S⋅√T⋅σ                


ν  (Vega) =


...

                                                                               ↪
                                                                               ↪
                                                                               ↪
                                                                               ↪
                                                                               ↪
                                                                               ↪
                                                                               ↪
     ⎛                         ⎛ 2⎞⎞ ⎛                                    ⎛ 2⎞ ↪
     ⎜  ⎛              2⎞      ⎜S ⎟⎟ ⎜       2     ⎛              2⎞      ⎜S ⎟ ↪
√2⋅S⋅⎜T⋅⎝-2⋅q + 2⋅r + σ ⎠ + log⎜──⎟⎟⋅⎜- 2⋅T⋅σ  + T⋅⎝-2⋅q + 2⋅r + σ ⎠ + log⎜──⎟ ↪
     ⎜                         ⎜ 2⎟⎟ ⎜                                    ⎜ 2⎟ ↪
     ⎝                         ⎝K ⎠⎠ ⎝                                    ⎝K ⎠ ↪
────────────────────────────


Θ  (Theta) =


                                                                               ↪
                                                                               ↪
                                                                               ↪
             ⎛                                    ⎛ 2⎞⎞                        ↪
             ⎜                                    ⎜S ⎟⎟                        ↪
             ⎜                              √2⋅log⎜──⎟⎟                        ↪
             ⎜                                    ⎜ 2⎟⎟                        ↪
     -T⋅r    ⎜√2⋅√T⋅q   √2⋅√T⋅r   √2⋅√T⋅σ         ⎝K ⎠⎟                    -T⋅ ↪
K⋅r⋅ℯ    ⋅erf⎜─────── - ─────── + ─────── - ──────────⎟        -T⋅r   S⋅q⋅ℯ    ↪
             ⎝  2⋅σ       2⋅σ        4        4⋅√T⋅σ  ⎠   K⋅r⋅ℯ                ↪
─────────────────────────────────────────────────────── - ───────── + ──────── ↪
                           2                                  2                ↪

↪                          


ρ  (Rho) =


    ⎛       ⎛   ⎛                          ⎛ 2⎞⎞⎞⎞      
    ⎜       ⎜   ⎜                   2      ⎜K ⎟⎟⎟⎟      
    ⎜       ⎜√2⋅⎜2⋅T⋅q - 2⋅T⋅r + T⋅σ  + log⎜──⎟⎟⎟⎟      
    ⎜       ⎜   ⎜                          ⎜ 2⎟⎟⎟⎟      
    ⎜       ⎜   ⎝                          ⎝S ⎠⎠⎟⎟  -T⋅r
K⋅T⋅⎜1 - erf⎜───────────────────────────────────⎟⎟⋅ℯ    
    ⎝       ⎝              4⋅√T⋅σ               ⎠⎠      
────────────────────────────────────────────────────────
                           2                            

## 5  Put-call parity

In [12]:
# Claim:  C - P = S·e^{-qT} - K·e^{-rT}
#
# This follows purely from N(x) + N(-x) = 1 (symmetry of the normal CDF),
# which SymPy derives from erf(-x) = -erf(x).

# First, verify the symmetry identity that underpins the result:
sym_check = sp.simplify(N(x) + N(-x) - 1)
print(f'N(x) + N(-x) - 1  =  {sym_check}')
assert sym_check == 0

# Now verify parity:
pcp_diff = sp.simplify(C - P - (S*sp.exp(-q*T) - K*sp.exp(-r*T)))
print(f'C - P - (S·e^(-qT) - K·e^(-rT))  =  {pcp_diff}')
assert pcp_diff == 0

# Put price follows immediately:
P_formula = K*sp.exp(-r*T)*N(-d2) - S*sp.exp(-q*T)*N(-d1)
print('\nP =')
display(P_formula)

N(x) + N(-x) - 1  =  0
C - P - (S·e^(-qT) - K·e^(-rT))  =  0

P =


  ⎛   ⎛   ⎛         ⎛          2⎞         ⎞⎞    ⎞                              ↪
  ⎜   ⎜   ⎜         ⎜         σ ⎟      ⎛S⎞⎟⎟    ⎟                              ↪
  ⎜   ⎜   ⎜       T⋅⎜-q + r + ──⎟ + log⎜─⎟⎟⎟    ⎟           ⎛       ⎛   ⎛  ⎛   ↪
  ⎜   ⎜   ⎜         ⎝         2 ⎠      ⎝K⎠⎟⎟    ⎟           ⎜       ⎜   ⎜  ⎜   ↪
  ⎜   ⎜√2⋅⎜√T⋅σ - ────────────────────────⎟⎟    ⎟           ⎜       ⎜√2⋅⎜T⋅⎜-q ↪
  ⎜   ⎜   ⎝                 √T⋅σ          ⎠⎟    ⎟           ⎜       ⎜   ⎝  ⎝   ↪
  ⎜erf⎜────────────────────────────────────⎟    ⎟           ⎜    erf⎜───────── ↪
  ⎜   ⎝                 2                  ⎠   1⎟  -T⋅r     ⎜1      ⎝          ↪
K⋅⎜───────────────────────────────────────── + ─⎟⋅ℯ     - S⋅⎜─ - ───────────── ↪
  ⎝                    2                       2⎠           ⎝2                 ↪

↪                             
↪                             
↪         2⎞         ⎞⎞⎞      
↪        σ ⎟      ⎛S⎞⎟⎟⎟      
↪  + r + ──⎟ + log⎜─⎟⎟⎟⎟      
↪        2 ⎠      ⎝K⎠⎠⎟⎟      
↪ ─